<a href="https://colab.research.google.com/github/Ariqueeezz/Enterprise-Brand-Sentiment-and-Campaign-Analytics-Pipeline/blob/epic%2FEBSCAP-9/Epic_EBSCAP_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving torch_ready_checkpoint.zip to torch_ready_checkpoint.zip


In [2]:
import zipfile
import os

torch_dir = "/content/torch_ready"
os.makedirs(torch_dir, exist_ok=True)

with zipfile.ZipFile("torch_ready_checkpoint.zip", "r") as zip_ref:
    zip_ref.extractall(torch_dir)

print("Isi folder torch_ready:")
print(os.listdir(torch_dir))

Isi folder torch_ready:
['word2idx.pkl', 'y_test.npy', 'X_val.npy', 'X_test.npy', 'y_val.npy', 'embedding_matrix.npy', 'X_train.npy', 'y_train.npy']


In [3]:
import numpy as np
import pickle
import torch
from torch.utils.data import Dataset, DataLoader

embedding_matrix = np.load(f"{torch_dir}/embedding_matrix.npy")
X_train = np.load(f"{torch_dir}/X_train.npy")
X_val = np.load(f"{torch_dir}/X_val.npy")
X_test = np.load(f"{torch_dir}/X_test.npy")
y_train = np.load(f"{torch_dir}/y_train.npy")
y_val = np.load(f"{torch_dir}/y_val.npy")
y_test = np.load(f"{torch_dir}/y_test.npy")

with open(f"{torch_dir}/word2idx.pkl", "rb") as f:
    word2idx = pickle.load(f)

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"Vocab size: {len(word2idx)}")

Embedding matrix shape: (47016, 100)
X_train shape: (489937, 100), y_train shape: (489937,)
Vocab size: 47016


# **EBSCAP-12: Rancang arsitektur model sekuensial (Bi-LSTM dengan Attention atau DistilBERT)**

In [5]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [6]:
class SentimentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(SentimentDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SentimentDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(SentimentDataset(X_test, y_test), batch_size=BATCH_SIZE)

xb, yb = next(iter(train_loader))
print(f"Sanity check batch shape: {xb.shape}, {yb.shape}")

Sanity check batch shape: torch.Size([64, 100]), torch.Size([64])


In [7]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)  # *2 karena bidirectional

    def forward(self, lstm_output, mask):
        scores = self.attn(lstm_output).squeeze(-1)          # [batch, seq_len]
        scores = scores.masked_fill(mask == 0, -1e9)          # abaikan posisi PAD
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)
        return context, weights

In [8]:
class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_classes=3,
                 num_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = Attention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.pad_idx = pad_idx

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        context, attn_weights = self.attention(lstm_out, mask)
        out = self.dropout(context)
        logits = self.fc(out)
        return logits, attn_weights

In [10]:
PAD_IDX = word2idx["<PAD>"]

model = BiLSTMAttentionClassifier(
    embedding_matrix=embedding_matrix,
    hidden_dim=128,
    num_classes=3,
    dropout=0.3,
    pad_idx=PAD_IDX
).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

BiLSTMAttentionClassifier(
  (embedding): Embedding(47016, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (attention): Attention(
    (attn): Linear(in_features=256, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)
Total params: 4,938,148 | Trainable: 4,938,148


In [11]:
xb, yb = xb.to(device), yb.to(device)

logits, attn_weights = model(xb)
print(f"Logits shape: {logits.shape}")                    # harus [64, 3]
print(f"Attention weights shape: {attn_weights.shape}")   # harus [64, 100]
print(f"Attention weights sum per sample (~1.0): {attn_weights.sum(dim=1)[:5]}")

Logits shape: torch.Size([64, 3])
Attention weights shape: torch.Size([64, 100])
Attention weights sum per sample (~1.0): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)
